# Two-Ring OT-tSNE Walkthrough

This notebook mirrors the ring workflow from `benchmarking.ipynb`, but the clean/original space now contains **two rings in each batch**:

- one ring centered at the origin,
- one same-radius ring shifted to the top-right,
- the same batch effect is applied across both ring components within each batch: batch 1 is translated by a vector and gets one extra noise dimension.

The pipeline shown below is:

1. generate clean and observed data,
2. compute the OT plan,
3. build the 4-block matrix,
4. apply **effective-k top-neighbor sparsification** using `sparsify_four_block_top_k` from `methods.py`,
5. normalize the effective-k graph with the `methods.py` graph normalizer (`binarize + symmetrize + normalize`),
6. run OT-tSNE,
7. compare against the **EOT `t=0`** embedding.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D

sys.path.insert(0, '../src')

from ne_aligned_utils.benchmarking import metric_triplet
from ne_aligned_utils.methods import (
    build_ot_tsne_four_block_matrix,
    compute_eot_embedding_from_plan,
    run_tsne_from_graph,
)
from ne_aligned_utils.methods import (
    normalize_ot_tsne_graph,
    resolve_effective_k_split,
    sparsify_four_block_top_k,
)
from ne_aligned_utils.methods import compute_ot_plan

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True


In [ ]:
seed = 0

n_per_ring_per_batch = 500
n0 = 2 * n_per_ring_per_batch
n1 = 2 * n_per_ring_per_batch

ring_radius = 2.5
ring_width = 0.3
large_radius = ring_radius
large_width = ring_width
small_radius = ring_radius
small_width = ring_width
ring_separation_gap = 12.0
ring_extent = ring_radius + ring_width
small_shift = (2 * ring_extent + ring_separation_gap, 2 * ring_extent + ring_separation_gap)
batch1_shift_2d = (10.0, 0.0)

eps_times = 1.0
tol = 1e-12
max_iter = int(1e4)

effective_k = 15
higher_dim_eps_times = 3.0
higher_dim_effective_k = 30
tsne_seed = 0
n_epochs = 200

eot_t = 0
embd_dim = 2
k_conc = 50
k_cross = 50

effective_k_split = resolve_effective_k_split(effective_k, n0=n0, n1=n1)
higher_dim_effective_k_split = resolve_effective_k_split(higher_dim_effective_k, n0=n0, n1=n1)
print(
    f'n_per_ring_per_batch={n_per_ring_per_batch}, n0={n0}, n1={n1}, '
    f'eps_times={eps_times:g}, effective_k={effective_k} '
    f'({effective_k_split.batch0_k}+{effective_k_split.batch1_k}), '
    f'EOT t={eot_t}, tsne_seed={tsne_seed}, n_epochs={n_epochs}'
)
print(
    f'3D/4D examples use eps_times={higher_dim_eps_times:g}, '
    f'effective_k={higher_dim_effective_k} '
    f'({higher_dim_effective_k_split.batch0_k}+{higher_dim_effective_k_split.batch1_k})'
)
print(
    f'ring_radius={ring_radius:g}, ring_width={ring_width:g}, '
    f'ring_separation_gap={ring_separation_gap:g}, batch1_shift_2d={batch1_shift_2d}'
)


def print_center_distance_margin(label, center0, center1, batch_shift):
    center0 = np.asarray(center0, dtype=float)
    center1 = np.asarray(center1, dtype=float)
    batch_shift = np.asarray(batch_shift, dtype=float)
    same_ring_distance = float(np.linalg.norm(batch_shift))
    cross_distances = (
        float(np.linalg.norm((center1 + batch_shift) - center0)),
        float(np.linalg.norm((center0 + batch_shift) - center1)),
    )
    nearest_cross_distance = min(cross_distances)
    print(
        f'{label} center distances: same-ring={same_ring_distance:.2f}, '
        f'nearest cross-ring={nearest_cross_distance:.2f}, '
        f'margin={nearest_cross_distance - same_ring_distance:.2f}'
    )


print_center_distance_margin('2D shifted rings', (0.0, 0.0), small_shift, batch1_shift_2d)


## Data Generation Helpers


In [ ]:
def sample_latent_ring(n, rng, base_radius, radial_width, center=(0.0, 0.0)):
    theta = np.sort(2 * np.pi * rng.random(n))
    radius = base_radius + radial_width * rng.random(n)
    x = center[0] + radius * np.cos(theta)
    y = center[1] + radius * np.sin(theta)
    return np.column_stack([x, y])


def split_ring_counts_evenly(n):
    if n < 2:
        raise ValueError(f'n must be at least 2 to split across two rings, got {n}.')

    n_small = n // 2
    n_large = n - n_small
    if n_large != n_small:
        raise ValueError(f'n must be even to split evenly across two rings, got {n}.')
    return n_large, n_small


def make_two_ring_batches(
    n0,
    n1,
    seed,
    large_radius,
    large_width,
    small_radius,
    small_width,
    small_shift,
    batch1_shift,
):
    seed_seq = np.random.SeedSequence(seed)
    rngs = [np.random.default_rng(s) for s in seed_seq.spawn(6)]

    n0_large, n0_small = split_ring_counts_evenly(n0)
    n1_large, n1_small = split_ring_counts_evenly(n1)

    clean0_large = sample_latent_ring(n0_large, rngs[0], large_radius, large_width)
    clean0_small = sample_latent_ring(
        n0_small,
        rngs[1],
        small_radius,
        small_width,
        center=small_shift,
    )
    clean1_large = sample_latent_ring(n1_large, rngs[2], large_radius, large_width)
    clean1_small = sample_latent_ring(
        n1_small,
        rngs[3],
        small_radius,
        small_width,
        center=small_shift,
    )

    clean0 = np.vstack([clean0_large, clean0_small])
    clean1 = np.vstack([clean1_large, clean1_small])

    component0 = np.concatenate([
        np.zeros(n0_large, dtype=int),
        np.ones(n0_small, dtype=int),
    ])
    component1 = np.concatenate([
        np.zeros(n1_large, dtype=int),
        np.ones(n1_small, dtype=int),
    ])

    batch1_shift = np.asarray(batch1_shift, dtype=float)
    if batch1_shift.shape != (2,):
        raise ValueError(f'batch1_shift must have shape (2,), got {batch1_shift.shape}.')

    obs0 = np.column_stack([clean0, np.zeros(clean0.shape[0], dtype=float)])
    obs1 = np.column_stack([
        clean1 + batch1_shift[None, :],
        rngs[4].random(clean1.shape[0]) - 0.5,
    ])

    batch_ids = np.concatenate([
        np.zeros(n0, dtype=int),
        np.ones(n1, dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])

    return {
        'clean0': clean0,
        'clean1': clean1,
        'obs0': obs0,
        'obs1': obs1,
        'component0': component0,
        'component1': component1,
        'batch_ids': batch_ids,
        'component_ids': component_ids,
    }


In [ ]:

BATCH_COLORS = {0: 'tab:red', 1: 'tab:blue'}
COMPONENT_COLORS = {0: 'tab:olive', 1: 'tab:purple'}
BATCH_NAMES = {0: 'batch 0', 1: 'batch 1'}
COMPONENT_NAMES = {0: 'origin ring', 1: 'shifted ring'}


def metric_box_text(metrics):
    return (
        f"conc={metrics['concordance']:.3f}\n"
        f"x-conc={metrics['cross_concordance']:.3f}\n"
        f"foscttm={metrics['foscttm']:.3f}"
    )


def _robust_positive_scale(values, quantile):
    values = np.asarray(values, dtype=float)
    positive_values = values[np.isfinite(values) & (values > 0)]
    if positive_values.size == 0:
        return 1.0

    scale = float(np.quantile(positive_values, quantile))
    if not np.isfinite(scale) or scale <= 0:
        scale = float(np.max(positive_values))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    return scale


def prepare_embedding_for_display(embedding, mode='isotropic', scale_quantile=0.95):
    emb = np.asarray(embedding, dtype=float)
    center = np.median(emb, axis=0, keepdims=True)
    centered = emb - center

    if mode == 'isotropic':
        radii = np.linalg.norm(centered, axis=1)
        scale = _robust_positive_scale(radii, scale_quantile)
        return centered / scale

    if mode == 'axiswise':
        scales = np.array(
            [
                _robust_positive_scale(np.abs(centered[:, axis]), scale_quantile)
                for axis in range(centered.shape[1])
            ],
            dtype=float,
        )
        scales[~np.isfinite(scales) | (scales <= 0)] = 1.0
        return centered / scales

    raise ValueError(f'Unknown display mode: {mode}')


def apply_robust_display_limits(ax, embedding, quantile=0.995, pad=0.08, square=False):
    emb = np.asarray(embedding, dtype=float)
    if emb.size == 0:
        return

    extents = np.quantile(np.abs(emb), quantile, axis=0)
    extents = np.asarray(extents, dtype=float)

    if square:
        limit = float(np.max(extents))
        if not np.isfinite(limit) or limit <= 0:
            limit = float(np.max(np.abs(emb))) if emb.size else 1.0
        if not np.isfinite(limit) or limit <= 0:
            limit = 1.0
        limit *= 1.0 + pad
        ax.set_xlim(-limit, limit)
        ax.set_ylim(-limit, limit)
        return

    fallback_extents = np.max(np.abs(emb), axis=0)
    extents = np.where(np.isfinite(extents) & (extents > 0), extents, fallback_extents)
    extents = np.where(np.isfinite(extents) & (extents > 0), extents, 1.0)
    extents = extents * (1.0 + pad)
    ax.set_xlim(-extents[0], extents[0])
    ax.set_ylim(-extents[1], extents[1])


def plot_latent_space(clean0, clean1, component0, component1, seed=0):
    clean = np.vstack([clean0, clean1])
    batch_ids = np.concatenate([
        np.zeros(clean0.shape[0], dtype=int),
        np.ones(clean1.shape[0], dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])
    perm = np.random.RandomState(seed).permutation(clean.shape[0])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    axes[0].scatter(clean[perm, 0], clean[perm, 1], c=batch_colors[perm], s=6, alpha=0.7)
    axes[0].set_title('Original latent space colored by batch')
    axes[0].set_xlabel('dim 1')
    axes[0].set_ylabel('dim 2')
    axes[0].set_aspect('equal')
    axes[0].legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=BATCH_NAMES[idx], markerfacecolor=color, markersize=8)
            for idx, color in BATCH_COLORS.items()
        ],
        loc='best',
    )

    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    axes[1].scatter(clean[perm, 0], clean[perm, 1], c=component_colors[perm], s=6, alpha=0.7)
    axes[1].set_title('Original latent space colored by ring')
    axes[1].set_xlabel('dim 1')
    axes[1].set_ylabel('dim 2')
    axes[1].set_aspect('equal')
    axes[1].legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=COMPONENT_NAMES[idx], markerfacecolor=color, markersize=8)
            for idx, color in COMPONENT_COLORS.items()
        ],
        loc='best',
    )
    plt.show()


def plot_observed_space(obs0, obs1, component0, component1, seed=0):
    obs = np.vstack([obs0, obs1])
    batch_ids = np.concatenate([
        np.zeros(obs0.shape[0], dtype=int),
        np.ones(obs1.shape[0], dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])
    perm = np.random.RandomState(seed).permutation(obs.shape[0])

    fig = plt.figure(figsize=(14, 5.5), constrained_layout=True)
    ax0 = fig.add_subplot(121, projection='3d')
    ax1 = fig.add_subplot(122, projection='3d')

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    ax0.scatter(obs[perm, 0], obs[perm, 1], obs[perm, 2], c=batch_colors[perm], s=2, alpha=0.35)
    ax0.set_title('Observed space with batch effect\n(colored by batch)')
    ax0.set_xlabel('dim 1')
    ax0.set_ylabel('dim 2')
    ax0.set_zlabel('dim 3')
    ax0.view_init(elev=28, azim=-60)
    ax0.legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=BATCH_NAMES[idx], markerfacecolor=color, markersize=8)
            for idx, color in BATCH_COLORS.items()
        ],
        loc='upper left',
    )

    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    ax1.scatter(obs[perm, 0], obs[perm, 1], obs[perm, 2], c=component_colors[perm], s=2, alpha=0.35)
    ax1.set_title('Observed space with batch effect\n(colored by ring)')
    ax1.set_xlabel('dim 1')
    ax1.set_ylabel('dim 2')
    ax1.set_zlabel('dim 3')
    ax1.view_init(elev=28, azim=-60)
    ax1.legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=COMPONENT_NAMES[idx], markerfacecolor=color, markersize=8)
            for idx, color in COMPONENT_COLORS.items()
        ],
        loc='upper left',
    )
    plt.show()


def plot_matrix(matrix, title, xlabel, ylabel, use_lognorm=False, figsize=(7, 5)):
    fig, ax = plt.subplots(1, 1, figsize=figsize, constrained_layout=True)
    display = np.array(matrix, copy=True)

    if use_lognorm:
        positive = display[display > 0]
        norm = None
        if positive.size > 0:
            vmin = max(float(np.quantile(positive, 0.01)), 1e-16)
            vmax = float(np.quantile(positive, 0.995))
            if vmax <= vmin:
                vmax = float(positive.max()) if float(positive.max()) > vmin else vmin * 10.0
            norm = LogNorm(vmin=vmin, vmax=vmax)
        display[display <= 0] = np.nan
        image = ax.imshow(display, cmap='magma', interpolation='nearest', aspect='auto', norm=norm)
    else:
        display = np.maximum(display, 0.0)
        image = ax.imshow(
            display,
            cmap='magma',
            interpolation='nearest',
            aspect='auto',
            vmin=0.0,
            vmax=float(display.max()) if display.size else 1.0,
        )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(False)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def plot_embedding(
    embedding,
    batch_ids,
    component_ids,
    title,
    metrics=None,
    seed=0,
    display_mode='isotropic',
    display_note=None,
    component_names=None,
    gradient_values=None,
    gradient_label=None,
    gradient_cmap='viridis',
):
    component_names = COMPONENT_NAMES if component_names is None else component_names
    embedding_display = prepare_embedding_for_display(embedding, mode=display_mode)
    perm = np.random.RandomState(seed).permutation(embedding_display.shape[0])

    has_gradient = gradient_values is not None
    n_panels = 3 if has_gradient else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5), constrained_layout=True)
    axes = np.atleast_1d(axes)

    title_suffix = '' if display_note is None else f'\n({display_note})'

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    axes[0].scatter(embedding_display[perm, 0], embedding_display[perm, 1], c=batch_colors[perm], s=3, alpha=0.35)
    axes[0].set_title(f'{title}{title_suffix}\n(colored by batch)')
    axes[0].set_aspect('equal')
    apply_robust_display_limits(axes[0], embedding_display, square=(display_mode == 'isotropic'))
    axes[0].axis('off')
    axes[0].legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=BATCH_NAMES[idx], markerfacecolor=color, markersize=8)
            for idx, color in BATCH_COLORS.items()
        ],
        loc='best',
    )
    if metrics is not None:
        axes[0].text(
            0.02,
            0.98,
            metric_box_text(metrics),
            transform=axes[0].transAxes,
            va='top',
            ha='left',
            fontsize=8,
            bbox={'boxstyle': 'round,pad=0.2', 'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
        )

    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    axes[1].scatter(embedding_display[perm, 0], embedding_display[perm, 1], c=component_colors[perm], s=3, alpha=0.35)
    axes[1].set_title(f'{title}{title_suffix}\n(colored by ring)')
    axes[1].set_aspect('equal')
    apply_robust_display_limits(axes[1], embedding_display, square=(display_mode == 'isotropic'))
    axes[1].axis('off')
    axes[1].legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=component_names.get(idx, f'component {idx}'), markerfacecolor=color, markersize=8)
            for idx, color in COMPONENT_COLORS.items()
        ],
        loc='best',
    )

    if has_gradient:
        gradient_values = np.asarray(gradient_values, dtype=float)
        if gradient_values.shape[0] != embedding_display.shape[0]:
            raise ValueError(
                'gradient_values must have the same number of entries as embedding rows, '
                f'got {gradient_values.shape[0]} and {embedding_display.shape[0]}.'
            )
        gradient_label = 'gradient value' if gradient_label is None else gradient_label
        gradient_scatter = axes[2].scatter(
            embedding_display[perm, 0],
            embedding_display[perm, 1],
            c=gradient_values[perm],
            s=3,
            alpha=0.35,
            cmap=gradient_cmap,
        )
        axes[2].set_title(f'{title}{title_suffix}\n(colored by {gradient_label})')
        axes[2].set_aspect('equal')
        apply_robust_display_limits(axes[2], embedding_display, square=(display_mode == 'isotropic'))
        axes[2].axis('off')
        fig.colorbar(gradient_scatter, ax=axes[2], fraction=0.046, pad=0.04, label=gradient_label)

    plt.show()

def print_metric_summary(label, metrics):
    print(
        f"{label}: conc={metrics['concordance']:.4f}, "
        f"x-conc={metrics['cross_concordance']:.4f}, "
        f"foscttm={metrics['foscttm']:.4f}"
    )



def embedding_span_summary(embedding):
    emb = np.asarray(embedding, dtype=float)
    mins = np.min(emb, axis=0)
    maxs = np.max(emb, axis=0)
    spans = maxs - mins
    ratio = spans[1] / spans[0] if spans[0] > 0 else np.nan
    return {
        'x_min': float(mins[0]),
        'x_max': float(maxs[0]),
        'y_min': float(mins[1]),
        'y_max': float(maxs[1]),
        'x_span': float(spans[0]),
        'y_span': float(spans[1]),
        'y_over_x': float(ratio),
    }



def plot_eot_ring_diagnostics(eot_embedding, n0, component0, component1, seed=0, t=0, component_names=None):
    component_names = COMPONENT_NAMES if component_names is None else component_names
    left_raw = np.asarray(eot_embedding[:n0], dtype=float)
    right_raw = np.asarray(eot_embedding[n0:], dtype=float)

    batch0 = np.zeros(component0.shape[0], dtype=int)
    batch1 = np.ones(component1.shape[0], dtype=int)

    ring_panels = []
    for ring_idx, ring_label in component_names.items():
        left_mask = component0 == ring_idx
        right_mask = component1 == ring_idx

        ring_raw = np.vstack([left_raw[left_mask], right_raw[right_mask]])
        ring_batches = np.concatenate([batch0[left_mask], batch1[right_mask]])
        ring_colors = np.array([BATCH_COLORS[int(v)] for v in ring_batches])
        ring_perm = np.random.RandomState(seed + int(ring_idx)).permutation(ring_raw.shape[0])
        ring_stats = embedding_span_summary(ring_raw)

        ring_panels.append(
            {
                'label': ring_label,
                'raw': ring_raw,
                'perm': ring_perm,
                'colors': ring_colors,
                'stats': ring_stats,
            }
        )

    def set_raw_zoom_limits(ax, emb, pad=0.12):
        mins = np.min(emb, axis=0)
        maxs = np.max(emb, axis=0)
        center = 0.5 * (mins + maxs)
        half_range = 0.5 * float(np.max(maxs - mins))
        if not np.isfinite(half_range) or half_range <= 0:
            half_range = 1e-12
        half_range *= 1.0 + pad
        ax.set_xlim(center[0] - half_range, center[0] + half_range)
        ax.set_ylim(center[1] - half_range, center[1] + half_range)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    panel_specs = (
        (axes[0], ring_panels[0]['raw'], ring_panels[0]['perm'], ring_panels[0]['colors'], f"{ring_panels[0]['label']} only (raw coordinates)", ring_panels[0]['stats'], 'overview'),
        (axes[1], ring_panels[1]['raw'], ring_panels[1]['perm'], ring_panels[1]['colors'], f"{ring_panels[1]['label']} only zoomed (raw coordinates)", ring_panels[1]['stats'], 'zoom'),
    )

    for ax, emb, perm, colors, title, stats, limit_mode in panel_specs:
        ax.scatter(emb[perm, 0], emb[perm, 1], c=colors[perm], s=6, alpha=0.45)
        ax.set_title(title)
        ax.set_aspect('equal')
        if limit_mode == 'zoom':
            set_raw_zoom_limits(ax, emb)
        else:
            apply_robust_display_limits(ax, emb, square=True)
        ax.set_xlabel('dim 1')
        ax.set_ylabel('dim 2')
        ax.text(
            0.02,
            0.98,
            (
                f"dx={stats['x_span']:.3e}\n"
                f"dy={stats['y_span']:.3e}\n"
                f"dy/dx={stats['y_over_x']:.3e}"
            ),
            transform=ax.transAxes,
            va='top',
            ha='left',
            fontsize=8,
            bbox={'boxstyle': 'round,pad=0.2', 'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
        )
        ax.grid(alpha=0.25)

    legend_handles = [
        Line2D([0], [0], marker='o', color='w', label=BATCH_NAMES[idx], markerfacecolor=color, markersize=8)
        for idx, color in BATCH_COLORS.items()
    ]
    axes[1].legend(handles=legend_handles, loc='upper right')

    fig.suptitle(
        f'EOT t={t} ring diagnostics: raw coordinates with shifted-ring zoom',
        fontsize=14,
    )
    plt.show()


## Generate Two-Ring Data


In [ ]:
data = make_two_ring_batches(
    n0=n0,
    n1=n1,
    seed=seed,
    large_radius=large_radius,
    large_width=large_width,
    small_radius=small_radius,
    small_width=small_width,
    small_shift=small_shift,
    batch1_shift=batch1_shift_2d,
)

print(
    f"clean0={data['clean0'].shape}, clean1={data['clean1'].shape}, "
    f"obs0={data['obs0'].shape}, obs1={data['obs1'].shape}"
)
print(
    'batch 0 counts:',
    {COMPONENT_NAMES[idx]: int((data['component0'] == idx).sum()) for idx in np.unique(data['component0'])},
)
print(
    'batch 1 counts:',
    {COMPONENT_NAMES[idx]: int((data['component1'] == idx).sum()) for idx in np.unique(data['component1'])},
)


In [ ]:
plot_latent_space(
    data['clean0'],
    data['clean1'],
    data['component0'],
    data['component1'],
    seed=seed,
)

plot_observed_space(
    data['obs0'],
    data['obs1'],
    data['component0'],
    data['component1'],
    seed=seed,
)


## OT Plan and 4-Block Graph Construction


In [ ]:
plan = compute_ot_plan(
    data['obs0'],
    data['obs1'],
    eps_times=eps_times,
    tol=tol,
    max_iter=max_iter,
)

print(f'OT plan shape={plan.shape}, total mass={plan.sum():.6f}')
plot_matrix(
    plan,
    title=f'Two-ring OT plan (eps_times={eps_times:g})',
    xlabel='batch 1 index',
    ylabel='batch 0 index',
    use_lognorm=True,
    figsize=(7.5, 5.5),
)

four_block = build_ot_tsne_four_block_matrix(plan)
four_block_nnz = int(np.count_nonzero(four_block))
print(
    f'4-block matrix shape={four_block.shape}, nnz={four_block_nnz}, '
    f'density={four_block_nnz / four_block.size:.4f}, sum={four_block.sum():.6f}'
)
plot_matrix(
    four_block,
    title='Full 4-block matrix from the OT plan',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=True,
    figsize=(7.5, 5.5),
)


## Effective-k Sparsification and Normalized Graph


In [ ]:
sparse_four_block, sparse_split = sparsify_four_block_top_k(
    four_block,
    n0=n0,
    n1=n1,
    effective_k=effective_k,
    binarize=True,
)
sparse_nnz = int(np.count_nonzero(sparse_four_block))
print(
    f'effective-k sparse 4-block support: effective_k={sparse_split.effective_k} '
    f'({sparse_split.batch0_k}+{sparse_split.batch1_k}), nnz={sparse_nnz}, '
    f'density={sparse_nnz / sparse_four_block.size:.4f}, sum={sparse_four_block.sum():.6f}'
)
plot_matrix(
    sparse_four_block,
    title=f'Effective-k sparsified 4-block support (k={effective_k})',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=False,
    figsize=(7.5, 5.5),
)

step5_graph = normalize_ot_tsne_graph(sparse_four_block)
step5_nnz = int(np.count_nonzero(step5_graph))
print(
    f'normalized effective-k graph: nnz={step5_nnz}, '
    f'density={step5_nnz / step5_graph.size:.4f}, sum={step5_graph.sum():.6f}'
)
plot_matrix(
    step5_graph,
    title='Normalized effective-k graph (binarized, symmetrized, normalized)',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=False,
    figsize=(7.5, 5.5),
)


## OT-tSNE Embedding


In [ ]:
ot_tsne_embedding = run_tsne_from_graph(
    step5_graph,
    seed=tsne_seed,
    n_epochs=n_epochs,
)
ot_tsne_metrics = metric_triplet(
    data['clean0'],
    data['clean1'],
    ot_tsne_embedding,
    k_conc=k_conc,
    k_cross=k_cross,
)

print_metric_summary('OT-tSNE', ot_tsne_metrics)
plot_embedding(
    ot_tsne_embedding,
    data['batch_ids'],
    data['component_ids'],
    title=f'OT-tSNE embedding from the effective-k graph\n(eps_times={eps_times:g}, k={effective_k})',
    metrics=ot_tsne_metrics,
    seed=tsne_seed,
)


## EOT `t=0` Embedding and Comparison


In [ ]:

eot_embedding = compute_eot_embedding_from_plan(
    plan,
    n0=n0,
    n1=n1,
    embd_dim=embd_dim,
    t=eot_t,
)
eot_metrics = metric_triplet(
    data['clean0'],
    data['clean1'],
    eot_embedding,
    k_conc=k_conc,
    k_cross=k_cross,
)

print_metric_summary(f'EOT t={eot_t}', eot_metrics)
plot_embedding(
    eot_embedding,
    data['batch_ids'],
    data['component_ids'],
    title=f'EOT embedding (t={eot_t})',
    metrics=eot_metrics,
    seed=tsne_seed,
    #display_mode='axiswise',
    #display_note='axis-wise robust scaling for display only',
)

batch_colors = np.array([BATCH_COLORS[int(v)] for v in data['batch_ids']])
perm = np.random.RandomState(tsne_seed).permutation(batch_colors.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, emb, title, metrics, display_mode in (
    (axes[0], ot_tsne_embedding, 'OT-tSNE', ot_tsne_metrics, 'isotropic'),
    (axes[1], eot_embedding, f'EOT t={eot_t}', eot_metrics, 'axiswise'),
):
    emb_display = prepare_embedding_for_display(emb, mode=display_mode)
    ax.scatter(emb_display[perm, 0], emb_display[perm, 1], c=batch_colors[perm], s=3, alpha=0.35)
    ax.set_title(title)
    ax.set_aspect('equal')
    apply_robust_display_limits(ax, emb_display, square=(display_mode == 'isotropic'))
    ax.axis('off')
    ax.text(
        0.02,
        0.98,
        metric_box_text(metrics),
        transform=ax.transAxes,
        va='top',
        ha='left',
        fontsize=8,
        bbox={'boxstyle': 'round,pad=0.2', 'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
    )

fig.suptitle('Batch-colored comparison: OT-tSNE vs EOT t=0\n(EOT uses axis-wise display scaling only)', fontsize=14)
plt.show()


plot_eot_ring_diagnostics(
    eot_embedding,
    n0=n0,
    component0=data['component0'],
    component1=data['component1'],
    seed=tsne_seed,
    t=eot_t,
)


## Orthogonal Floating Ring in 3D with a 4D Batch-1 Noise Dimension

This variant keeps the same-radius clean/original geometry in **3D**. The `x-y` ring lies in the `x-y` plane at `z=0`, and the vertical ring is a separated `y-z` circle shifted outward and upward from the `x-y` ring. The observed data are **4D**: batch 0 keeps the clean coordinates with fourth coordinate `0`, while batch 1 is translated by a clean-space vector and gets an extra noisy fourth coordinate. The higher-dimensional examples use a sharper OT plan and denser effective-k graph than the 2D walkthrough so the local cyclic support of both rings survives the OT-tSNE step.


In [ ]:
FLOATING_COMPONENT_NAMES = {0: 'xy ring', 1: 'vertical floating ring'}
floating_ring_gap = ring_separation_gap
floating_ring_center = (
    0.0,
    2 * ring_extent + floating_ring_gap,
    ring_extent + floating_ring_gap,
)
fourth_noise_width = 1.0
batch1_shift_3d = (10.0, 0.0, 0.0)

print_center_distance_margin('3D floating rings', (0.0, 0.0, 0.0), floating_ring_center, batch1_shift_3d)


def _ring_plane_axes(plane):
    axes = {
        'xy': (np.array([1.0, 0.0, 0.0]), np.array([0.0, 1.0, 0.0])),
        'xz': (np.array([1.0, 0.0, 0.0]), np.array([0.0, 0.0, 1.0])),
        'yz': (np.array([0.0, 1.0, 0.0]), np.array([0.0, 0.0, 1.0])),
    }
    if plane not in axes:
        raise ValueError(f'Unknown ring plane: {plane!r}. Use one of {tuple(axes)}.')
    return axes[plane]


def sample_latent_ring_3d(n, rng, base_radius, radial_width, center=(0.0, 0.0, 0.0), plane='xy'):
    theta = np.sort(2 * np.pi * rng.random(n))
    radius = base_radius + radial_width * rng.random(n)
    axis_a, axis_b = _ring_plane_axes(plane)
    center = np.asarray(center, dtype=float)
    offsets = radius[:, None] * (
        np.cos(theta)[:, None] * axis_a[None, :] + np.sin(theta)[:, None] * axis_b[None, :]
    )
    return center[None, :] + offsets


def sample_centered_noise(n, rng, width):
    if n <= 0:
        return np.empty(0, dtype=float)

    noise = rng.random(n) - 0.5
    if n > 1:
        noise_min = float(noise.min())
        noise_span = float(noise.max() - noise_min)
        if noise_span > 0:
            noise = (noise - noise_min) / noise_span - 0.5
    return width * noise


def make_floating_two_ring_batches(
    n0,
    n1,
    seed,
    large_radius,
    large_width,
    small_radius,
    small_width,
    small_center,
    fourth_noise_width=1.0,
    batch1_shift=(10.0, 0.0, 0.0),
):
    seed_seq = np.random.SeedSequence(seed)
    rngs = [np.random.default_rng(s) for s in seed_seq.spawn(6)]

    n0_large, n0_small = split_ring_counts_evenly(n0)
    n1_large, n1_small = split_ring_counts_evenly(n1)

    clean0_large = sample_latent_ring_3d(n0_large, rngs[0], large_radius, large_width)
    clean0_small = sample_latent_ring_3d(
        n0_small,
        rngs[1],
        small_radius,
        small_width,
        center=small_center,
        plane='yz',
    )
    clean1_large = sample_latent_ring_3d(n1_large, rngs[2], large_radius, large_width)
    clean1_small = sample_latent_ring_3d(
        n1_small,
        rngs[3],
        small_radius,
        small_width,
        center=small_center,
        plane='yz',
    )

    clean0 = np.vstack([clean0_large, clean0_small])
    clean1 = np.vstack([clean1_large, clean1_small])

    component0 = np.concatenate([
        np.zeros(n0_large, dtype=int),
        np.ones(n0_small, dtype=int),
    ])
    component1 = np.concatenate([
        np.zeros(n1_large, dtype=int),
        np.ones(n1_small, dtype=int),
    ])

    batch1_shift = np.asarray(batch1_shift, dtype=float)
    if batch1_shift.shape != (3,):
        raise ValueError(f'batch1_shift must have shape (3,), got {batch1_shift.shape}.')

    obs0 = np.column_stack([clean0, np.zeros(clean0.shape[0], dtype=float)])
    obs1 = np.column_stack([
        clean1 + batch1_shift[None, :],
        sample_centered_noise(clean1.shape[0], rngs[4], fourth_noise_width),
    ])

    batch_ids = np.concatenate([
        np.zeros(n0, dtype=int),
        np.ones(n1, dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])

    return {
        'clean0': clean0,
        'clean1': clean1,
        'obs0': obs0,
        'obs1': obs1,
        'component0': component0,
        'component1': component1,
        'batch_ids': batch_ids,
        'component_ids': component_ids,
    }


def _set_3d_box_aspect(ax, points):
    spans = np.ptp(np.asarray(points, dtype=float), axis=0)
    spans = np.where(np.isfinite(spans) & (spans > 0), spans, 1.0)
    ax.set_box_aspect(spans)


def plot_latent_space_3d(clean0, clean1, component0, component1, seed=0, component_names=None):
    component_names = FLOATING_COMPONENT_NAMES if component_names is None else component_names
    clean = np.vstack([clean0, clean1])
    batch_ids = np.concatenate([
        np.zeros(clean0.shape[0], dtype=int),
        np.ones(clean1.shape[0], dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])
    perm = np.random.RandomState(seed).permutation(clean.shape[0])

    fig = plt.figure(figsize=(14, 5.5), constrained_layout=True)
    ax0 = fig.add_subplot(121, projection='3d')
    ax1 = fig.add_subplot(122, projection='3d')

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    ax0.scatter(clean[perm, 0], clean[perm, 1], clean[perm, 2], c=batch_colors[perm], s=2, alpha=0.65)
    ax0.set_title('3D original space colored by batch')
    ax0.set_xlabel('dim 1')
    ax0.set_ylabel('dim 2')
    ax0.set_zlabel('dim 3')
    ax0.view_init(elev=24, azim=-58)
    _set_3d_box_aspect(ax0, clean)
    ax0.legend(
        handles=[
            Line2D([0], [0], marker='o', color='w', label=BATCH_NAMES[idx], markerfacecolor=color, markersize=8)
            for idx, color in BATCH_COLORS.items()
        ],
        loc='upper left',
    )

    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    ax1.scatter(clean[perm, 0], clean[perm, 1], clean[perm, 2], c=component_colors[perm], s=2, alpha=0.65)
    ax1.set_title('3D original space colored by ring')
    ax1.set_xlabel('dim 1')
    ax1.set_ylabel('dim 2')
    ax1.set_zlabel('dim 3')
    ax1.view_init(elev=24, azim=-58)
    _set_3d_box_aspect(ax1, clean)
    ax1.legend(
        handles=[
            Line2D(
                [0],
                [0],
                marker='o',
                color='w',
                label=component_names.get(idx, f'component {idx}'),
                markerfacecolor=color,
                markersize=8,
            )
            for idx, color in COMPONENT_COLORS.items()
        ],
        loc='upper left',
    )
    plt.show()


def plot_observed_space_4d_projections(obs0, obs1, component0, component1, seed=0, component_names=None):
    component_names = FLOATING_COMPONENT_NAMES if component_names is None else component_names
    obs = np.vstack([obs0, obs1])
    batch_ids = np.concatenate([
        np.zeros(obs0.shape[0], dtype=int),
        np.ones(obs1.shape[0], dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])
    perm = np.random.RandomState(seed).permutation(obs.shape[0])

    def set_projection_box_aspect(ax, points, min_fraction=0.18):
        spans = np.ptp(np.asarray(points, dtype=float), axis=0)
        spans = np.where(np.isfinite(spans) & (spans > 0), spans, 1.0)
        max_span = float(np.max(spans))
        spans = np.maximum(spans, min_fraction * max_span)
        ax.set_box_aspect(spans)

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    color_specs = (
        (batch_colors, 'batch', BATCH_NAMES, BATCH_COLORS),
        (component_colors, 'ring', component_names, COMPONENT_COLORS),
    )
    projection_specs = (
        ((0, 1, 2), 'dims 1-2-3 clean geometry', 24, -58),
        ((0, 1, 3), 'dims 1-2-4 batch noise', 24, -58),
        ((1, 2, 3), 'dims 2-3-4 vertical/noise', 24, -58),
    )

    fig = plt.figure(figsize=(21, 10), constrained_layout=True)
    for row_idx, (colors, color_label, names, palette) in enumerate(color_specs):
        for col_idx, (dims, projection_label, elev, azim) in enumerate(projection_specs):
            ax = fig.add_subplot(2, 3, row_idx * 3 + col_idx + 1, projection='3d')
            projected = obs[:, dims]
            ax.scatter(
                projected[perm, 0],
                projected[perm, 1],
                projected[perm, 2],
                c=colors[perm],
                s=2,
                alpha=0.35,
            )
            ax.set_title(f'Observed 4D projected to {projection_label}\n(colored by {color_label})')
            ax.set_xlabel(f'dim {dims[0] + 1}')
            ax.set_ylabel(f'dim {dims[1] + 1}')
            ax.set_zlabel(f'dim {dims[2] + 1}')
            ax.view_init(elev=elev, azim=azim)
            set_projection_box_aspect(ax, projected)
            ax.legend(
                handles=[
                    Line2D(
                        [0],
                        [0],
                        marker='o',
                        color='w',
                        label=names.get(idx, f'component {idx}'),
                        markerfacecolor=color,
                        markersize=8,
                    )
                    for idx, color in palette.items()
                ],
                loc='upper left',
            )
    plt.show()


In [ ]:
floating_data = make_floating_two_ring_batches(
    n0=n0,
    n1=n1,
    seed=seed,
    large_radius=large_radius,
    large_width=large_width,
    small_radius=small_radius,
    small_width=small_width,
    small_center=floating_ring_center,
    fourth_noise_width=fourth_noise_width,
    batch1_shift=batch1_shift_3d,
)

print(
    f"clean0={floating_data['clean0'].shape}, clean1={floating_data['clean1'].shape}, "
    f"obs0={floating_data['obs0'].shape}, obs1={floating_data['obs1'].shape}"
)
print(
    'batch 0 counts:',
    {
        FLOATING_COMPONENT_NAMES[idx]: int((floating_data['component0'] == idx).sum())
        for idx in np.unique(floating_data['component0'])
    },
)
print(
    'batch 1 counts:',
    {
        FLOATING_COMPONENT_NAMES[idx]: int((floating_data['component1'] == idx).sum())
        for idx in np.unique(floating_data['component1'])
    },
)
print(
    'observed dim 1 ranges:',
    f"batch 0 [{floating_data['obs0'][:, 0].min():.2f}, {floating_data['obs0'][:, 0].max():.2f}], "
    f"batch 1 [{floating_data['obs1'][:, 0].min():.2f}, {floating_data['obs1'][:, 0].max():.2f}]",
)

print(
    'batch 1 fourth-coordinate noise span:',
    f"{np.ptp(floating_data['obs1'][:, 3]):.3f}",
)

plot_latent_space_3d(
    floating_data['clean0'],
    floating_data['clean1'],
    floating_data['component0'],
    floating_data['component1'],
    seed=seed,
)
plot_observed_space_4d_projections(
    floating_data['obs0'],
    floating_data['obs1'],
    floating_data['component0'],
    floating_data['component1'],
    seed=seed,
)


### OT Plan, Effective-k Graph, and Embeddings for the 4D Noise Variant


In [ ]:
floating_plan = compute_ot_plan(
    floating_data['obs0'],
    floating_data['obs1'],
    eps_times=higher_dim_eps_times,
    tol=tol,
    max_iter=max_iter,
)

print(f'orthogonal floating OT plan shape={floating_plan.shape}, total mass={floating_plan.sum():.6f}')
plot_matrix(
    floating_plan,
    title=f'Orthogonal floating 3D/4D two-ring OT plan (eps_times={higher_dim_eps_times:g})',
    xlabel='batch 1 index',
    ylabel='batch 0 index',
    use_lognorm=True,
    figsize=(7.5, 5.5),
)

floating_four_block = build_ot_tsne_four_block_matrix(floating_plan)
floating_four_block_nnz = int(np.count_nonzero(floating_four_block))
print(
    f'orthogonal floating 3D/4D 4-block matrix shape={floating_four_block.shape}, '
    f'nnz={floating_four_block_nnz}, '
    f'density={floating_four_block_nnz / floating_four_block.size:.4f}, '
    f'sum={floating_four_block.sum():.6f}'
)
plot_matrix(
    floating_four_block,
    title='Orthogonal floating 3D/4D variant full 4-block matrix from the OT plan',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=True,
    figsize=(7.5, 5.5),
)

floating_sparse_four_block, floating_sparse_split = sparsify_four_block_top_k(
    floating_four_block,
    n0=n0,
    n1=n1,
    effective_k=higher_dim_effective_k,
    binarize=True,
)
floating_sparse_nnz = int(np.count_nonzero(floating_sparse_four_block))
print(
    f'orthogonal floating effective-k sparse 4-block support: '
    f'effective_k={floating_sparse_split.effective_k} '
    f'({floating_sparse_split.batch0_k}+{floating_sparse_split.batch1_k}), '
    f'nnz={floating_sparse_nnz}, '
    f'density={floating_sparse_nnz / floating_sparse_four_block.size:.4f}, '
    f'sum={floating_sparse_four_block.sum():.6f}'
)
plot_matrix(
    floating_sparse_four_block,
    title=f'Orthogonal floating 3D/4D effective-k support (k={higher_dim_effective_k})',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=False,
    figsize=(7.5, 5.5),
)

floating_step5_graph = normalize_ot_tsne_graph(floating_sparse_four_block)
floating_step5_nnz = int(np.count_nonzero(floating_step5_graph))
print(
    f'orthogonal floating 3D/4D normalized effective-k graph: nnz={floating_step5_nnz}, '
    f'density={floating_step5_nnz / floating_step5_graph.size:.4f}, '
    f'sum={floating_step5_graph.sum():.6f}'
)
plot_matrix(
    floating_step5_graph,
    title='Orthogonal floating 3D/4D normalized effective-k graph',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=False,
    figsize=(7.5, 5.5),
)


In [ ]:
floating_ot_tsne_embedding = run_tsne_from_graph(
    floating_step5_graph,
    seed=tsne_seed,
    n_epochs=n_epochs,
)
floating_ot_tsne_metrics = metric_triplet(
    floating_data['clean0'],
    floating_data['clean1'],
    floating_ot_tsne_embedding,
    k_conc=k_conc,
    k_cross=k_cross,
)

print_metric_summary('Orthogonal floating OT-tSNE', floating_ot_tsne_metrics)
plot_embedding(
    floating_ot_tsne_embedding,
    floating_data['batch_ids'],
    floating_data['component_ids'],
    title=f'Orthogonal floating 3D/4D OT-tSNE embedding\n(eps_times={higher_dim_eps_times:g}, k={higher_dim_effective_k})',
    metrics=floating_ot_tsne_metrics,
    seed=tsne_seed,
    component_names=FLOATING_COMPONENT_NAMES,
)

floating_eot_embedding = compute_eot_embedding_from_plan(
    floating_plan,
    n0=n0,
    n1=n1,
    embd_dim=embd_dim,
    t=eot_t,
)
floating_eot_metrics = metric_triplet(
    floating_data['clean0'],
    floating_data['clean1'],
    floating_eot_embedding,
    k_conc=k_conc,
    k_cross=k_cross,
)

print_metric_summary(f'Orthogonal floating EOT t={eot_t}', floating_eot_metrics)
floating_clean_dim2 = np.concatenate([
    floating_data['clean0'][:, 1],
    floating_data['clean1'][:, 1],
])
plot_embedding(
    floating_eot_embedding,
    floating_data['batch_ids'],
    floating_data['component_ids'],
    title=f'Orthogonal floating 3D/4D EOT embedding (t={eot_t})',
    metrics=floating_eot_metrics,
    seed=tsne_seed,
    display_mode='axiswise',
    display_note='axis-wise robust scaling for display only',
    component_names=FLOATING_COMPONENT_NAMES,
    gradient_values=floating_clean_dim2,
    gradient_label='clean dim 2',
)

floating_small_mask = floating_data['component_ids'] == 1
floating_small_indices = np.flatnonzero(floating_small_mask)
floating_small_perm = floating_small_indices[
    np.random.RandomState(tsne_seed).permutation(floating_small_indices.size)
]
floating_eot_display = prepare_embedding_for_display(floating_eot_embedding, mode='axiswise')
floating_small_display = floating_eot_display[floating_small_indices]
floating_small_dim2 = floating_clean_dim2[floating_small_indices]

print(
    'floating ring clean dim 2 span:',
    f"[{floating_small_dim2.min():.3f}, {floating_small_dim2.max():.3f}], "
    f"width={np.ptp(floating_small_dim2):.3f}",
)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
small_scatter = ax.scatter(
    floating_eot_display[floating_small_perm, 0],
    floating_eot_display[floating_small_perm, 1],
    c=floating_clean_dim2[floating_small_perm],
    s=8,
    alpha=0.65,
    cmap='viridis',
    vmin=float(floating_small_dim2.min()),
    vmax=float(floating_small_dim2.max()),
)
ax.set_title(
    f'Vertical floating ring only in EOT embedding (t={eot_t})\n'
    '(colored by clean dim 2; vertical-ring color scale)'
)
ax.set_aspect('equal', adjustable='box')
small_x_span = np.ptp(floating_small_display[:, 0])
small_y_span = np.ptp(floating_small_display[:, 1])
small_span = max(float(small_x_span), float(small_y_span), 1e-9)
small_x_pad = 0.08 * small_span
small_y_pad = 0.08 * small_span
ax.set_xlim(floating_small_display[:, 0].min() - small_x_pad, floating_small_display[:, 0].max() + small_x_pad)
ax.set_ylim(floating_small_display[:, 1].min() - small_y_pad, floating_small_display[:, 1].max() + small_y_pad)
ax.axis('off')
fig.colorbar(small_scatter, ax=ax, fraction=0.046, pad=0.04, label='clean dim 2')
plt.show()

floating_batch_colors = np.array([BATCH_COLORS[int(v)] for v in floating_data['batch_ids']])
floating_perm = np.random.RandomState(tsne_seed).permutation(floating_batch_colors.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, emb, title, metrics, display_mode in (
    (axes[0], floating_ot_tsne_embedding, 'Orthogonal floating OT-tSNE', floating_ot_tsne_metrics, 'isotropic'),
    (axes[1], floating_eot_embedding, f'Orthogonal floating EOT t={eot_t}', floating_eot_metrics, 'axiswise'),
):
    emb_display = prepare_embedding_for_display(emb, mode=display_mode)
    ax.scatter(
        emb_display[floating_perm, 0],
        emb_display[floating_perm, 1],
        c=floating_batch_colors[floating_perm],
        s=3,
        alpha=0.35,
    )
    ax.set_title(title)
    ax.set_aspect('equal')
    apply_robust_display_limits(ax, emb_display, square=(display_mode == 'isotropic'))
    ax.axis('off')
    ax.text(
        0.02,
        0.98,
        metric_box_text(metrics),
        transform=ax.transAxes,
        va='top',
        ha='left',
        fontsize=8,
        bbox={'boxstyle': 'round,pad=0.2', 'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
    )

fig.suptitle(
    'Batch-colored comparison: orthogonal floating 3D/4D OT-tSNE vs EOT t=0\n'
    '(EOT uses axis-wise display scaling only)',
    fontsize=14,
)
plt.show()

plot_eot_ring_diagnostics(
    floating_eot_embedding,
    n0=n0,
    component0=floating_data['component0'],
    component1=floating_data['component1'],
    seed=tsne_seed,
    t=eot_t,
    component_names=FLOATING_COMPONENT_NAMES,
)


## Orthogonal Plane Circles in 4D with a 5D Batch-1 Noise Dimension

This variant keeps the same-radius clean/original geometry in **4D**: one circle lives only in dims 1-2 and the other circle lives only in dims 3-4, shifted away from the first circle. The observed data are then embedded in **5D**: batch 0 keeps the clean coordinates with dim 5 = 0, while batch 1 is translated by a clean-space vector and gets the extra noise coordinate. It uses the same sharper higher-dimensional OT-tSNE settings as the 3D variant.

In the 2D projection panels below, the circle that lives in the other orthogonal plane collapses to a point: the dim 3-4 circle has coordinates `(0, 0)` in dims 1-2, and the dim 1-2 circle has coordinates `(0, 0)` in dims 3-4.


In [ ]:
PLANE4D_COMPONENT_NAMES = {0: 'dim 1-2 circle', 1: 'dim 3-4 circle'}
plane4d_ring_center = (0.0, 0.0, 2 * ring_extent + ring_separation_gap, 2 * ring_extent + ring_separation_gap)
plane4d_fifth_noise_width = 1.0
plane4d_batch1_shift = (10.0, 0.0, 0.0, 0.0)

print_center_distance_margin(
    '4D orthogonal-plane circles',
    (0.0, 0.0, 0.0, 0.0),
    plane4d_ring_center,
    plane4d_batch1_shift,
)


def sample_latent_ring_4d(n, rng, base_radius, radial_width, dims=(0, 1), center=None):
    if len(dims) != 2:
        raise ValueError(f'dims must contain exactly two axes, got {dims!r}.')
    if len(set(dims)) != 2 or any(axis < 0 or axis >= 4 for axis in dims):
        raise ValueError(f'dims must be two distinct 4D axes, got {dims!r}.')

    theta = np.sort(2 * np.pi * rng.random(n))
    radius = base_radius + radial_width * rng.random(n)
    center = np.zeros(4, dtype=float) if center is None else np.asarray(center, dtype=float)
    if center.shape != (4,):
        raise ValueError(f'center must have shape (4,), got {center.shape}.')

    points = np.repeat(center[None, :], n, axis=0)
    points[:, dims[0]] += radius * np.cos(theta)
    points[:, dims[1]] += radius * np.sin(theta)
    return points


def make_plane4d_two_ring_batches(
    n0,
    n1,
    seed,
    large_radius,
    large_width,
    small_radius,
    small_width,
    small_center,
    fifth_noise_width=1.0,
    batch1_shift=(10.0, 0.0, 0.0, 0.0),
):
    seed_seq = np.random.SeedSequence(seed)
    rngs = [np.random.default_rng(s) for s in seed_seq.spawn(6)]

    n0_large, n0_small = split_ring_counts_evenly(n0)
    n1_large, n1_small = split_ring_counts_evenly(n1)

    clean0_large = sample_latent_ring_4d(n0_large, rngs[0], large_radius, large_width, dims=(0, 1))
    clean0_small = sample_latent_ring_4d(
        n0_small,
        rngs[1],
        small_radius,
        small_width,
        dims=(2, 3),
        center=small_center,
    )
    clean1_large = sample_latent_ring_4d(n1_large, rngs[2], large_radius, large_width, dims=(0, 1))
    clean1_small = sample_latent_ring_4d(
        n1_small,
        rngs[3],
        small_radius,
        small_width,
        dims=(2, 3),
        center=small_center,
    )

    clean0 = np.vstack([clean0_large, clean0_small])
    clean1 = np.vstack([clean1_large, clean1_small])

    component0 = np.concatenate([
        np.zeros(n0_large, dtype=int),
        np.ones(n0_small, dtype=int),
    ])
    component1 = np.concatenate([
        np.zeros(n1_large, dtype=int),
        np.ones(n1_small, dtype=int),
    ])

    batch1_shift = np.asarray(batch1_shift, dtype=float)
    if batch1_shift.shape != (4,):
        raise ValueError(f'batch1_shift must have shape (4,), got {batch1_shift.shape}.')

    obs0 = np.column_stack([clean0, np.zeros(clean0.shape[0], dtype=float)])
    obs1 = np.column_stack([
        clean1 + batch1_shift[None, :],
        sample_centered_noise(clean1.shape[0], rngs[4], fifth_noise_width),
    ])

    batch_ids = np.concatenate([
        np.zeros(n0, dtype=int),
        np.ones(n1, dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])

    return {
        'clean0': clean0,
        'clean1': clean1,
        'obs0': obs0,
        'obs1': obs1,
        'component0': component0,
        'component1': component1,
        'batch_ids': batch_ids,
        'component_ids': component_ids,
    }


def plot_clean_plane4d_projections(clean0, clean1, component0, component1, seed=0, component_names=None):
    component_names = PLANE4D_COMPONENT_NAMES if component_names is None else component_names
    clean = np.vstack([clean0, clean1])
    batch_ids = np.concatenate([
        np.zeros(clean0.shape[0], dtype=int),
        np.ones(clean1.shape[0], dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])
    perm = np.random.RandomState(seed).permutation(clean.shape[0])

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    color_specs = (
        (batch_colors, 'batch', BATCH_NAMES, BATCH_COLORS),
        (component_colors, 'ring', component_names, COMPONENT_COLORS),
    )
    projection_specs = (
        ((0, 1), 'dims 1-2 plane'),
        ((2, 3), 'dims 3-4 plane'),
    )

    fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)
    for row_idx, (colors, color_label, names, palette) in enumerate(color_specs):
        for col_idx, (dims, projection_label) in enumerate(projection_specs):
            ax = axes[row_idx, col_idx]
            projected = clean[:, dims]
            ax.scatter(projected[perm, 0], projected[perm, 1], c=colors[perm], s=3, alpha=0.5)
            ax.set_title(f'Clean 4D projected to {projection_label}\n(colored by {color_label})')
            ax.set_xlabel(f'dim {dims[0] + 1}')
            ax.set_ylabel(f'dim {dims[1] + 1}')
            ax.set_aspect('equal', adjustable='box')
            ax.legend(
                handles=[
                    Line2D(
                        [0],
                        [0],
                        marker='o',
                        color='w',
                        label=names.get(idx, f'component {idx}'),
                        markerfacecolor=color,
                        markersize=8,
                    )
                    for idx, color in palette.items()
                ],
                loc='best',
            )
    plt.show()


def plot_observed_space_5d_projections(obs0, obs1, component0, component1, seed=0, component_names=None):
    component_names = PLANE4D_COMPONENT_NAMES if component_names is None else component_names
    obs = np.vstack([obs0, obs1])
    batch_ids = np.concatenate([
        np.zeros(obs0.shape[0], dtype=int),
        np.ones(obs1.shape[0], dtype=int),
    ])
    component_ids = np.concatenate([component0, component1])
    perm = np.random.RandomState(seed).permutation(obs.shape[0])

    def set_projection_box_aspect(ax, points, min_fraction=0.22):
        spans = np.ptp(np.asarray(points, dtype=float), axis=0)
        spans = np.where(np.isfinite(spans) & (spans > 0), spans, 1.0)
        max_span = float(np.max(spans))
        spans = np.maximum(spans, min_fraction * max_span)
        ax.set_box_aspect(spans)

    batch_colors = np.array([BATCH_COLORS[int(v)] for v in batch_ids])
    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in component_ids])
    color_specs = (
        (batch_colors, 'batch', BATCH_NAMES, BATCH_COLORS),
        (component_colors, 'ring', component_names, COMPONENT_COLORS),
    )
    projection_specs = (
        ((0, 1, 4), 'dims 1-2-5 first circle/noise', 22, -62),
        ((2, 3, 4), 'dims 3-4-5 second circle/noise', 22, -62),
        ((0, 2, 4), 'dims 1-3-5 shift/noise', 22, -62),
    )

    fig = plt.figure(figsize=(21, 10), constrained_layout=True)
    for row_idx, (colors, color_label, names, palette) in enumerate(color_specs):
        for col_idx, (dims, projection_label, elev, azim) in enumerate(projection_specs):
            ax = fig.add_subplot(2, 3, row_idx * 3 + col_idx + 1, projection='3d')
            projected = obs[:, dims]
            ax.scatter(
                projected[perm, 0],
                projected[perm, 1],
                projected[perm, 2],
                c=colors[perm],
                s=2,
                alpha=0.35,
            )
            ax.set_title(f'Observed 5D projected to {projection_label}\n(colored by {color_label})')
            ax.set_xlabel(f'dim {dims[0] + 1}')
            ax.set_ylabel(f'dim {dims[1] + 1}')
            ax.set_zlabel(f'dim {dims[2] + 1}')
            ax.view_init(elev=elev, azim=azim)
            set_projection_box_aspect(ax, projected)
            ax.legend(
                handles=[
                    Line2D(
                        [0],
                        [0],
                        marker='o',
                        color='w',
                        label=names.get(idx, f'component {idx}'),
                        markerfacecolor=color,
                        markersize=8,
                    )
                    for idx, color in palette.items()
                ],
                loc='upper left',
            )
    plt.show()


In [ ]:
plane4d_data = make_plane4d_two_ring_batches(
    n0=n0,
    n1=n1,
    seed=seed,
    large_radius=large_radius,
    large_width=large_width,
    small_radius=small_radius,
    small_width=small_width,
    small_center=plane4d_ring_center,
    fifth_noise_width=plane4d_fifth_noise_width,
    batch1_shift=plane4d_batch1_shift,
)

print(
    f"clean0={plane4d_data['clean0'].shape}, clean1={plane4d_data['clean1'].shape}, "
    f"obs0={plane4d_data['obs0'].shape}, obs1={plane4d_data['obs1'].shape}"
)
print(
    'batch 0 counts:',
    {
        PLANE4D_COMPONENT_NAMES[idx]: int((plane4d_data['component0'] == idx).sum())
        for idx in np.unique(plane4d_data['component0'])
    },
)
print(
    'batch 1 counts:',
    {
        PLANE4D_COMPONENT_NAMES[idx]: int((plane4d_data['component1'] == idx).sum())
        for idx in np.unique(plane4d_data['component1'])
    },
)
print(
    'observed dim 1 ranges:',
    f"batch 0 [{plane4d_data['obs0'][:, 0].min():.2f}, {plane4d_data['obs0'][:, 0].max():.2f}], "
    f"batch 1 [{plane4d_data['obs1'][:, 0].min():.2f}, {plane4d_data['obs1'][:, 0].max():.2f}]",
)
print(
    'batch 1 fifth-coordinate noise span:',
    f"{np.ptp(plane4d_data['obs1'][:, 4]):.3f}",
)

plot_clean_plane4d_projections(
    plane4d_data['clean0'],
    plane4d_data['clean1'],
    plane4d_data['component0'],
    plane4d_data['component1'],
    seed=seed,
)
plot_observed_space_5d_projections(
    plane4d_data['obs0'],
    plane4d_data['obs1'],
    plane4d_data['component0'],
    plane4d_data['component1'],
    seed=seed,
)


### OT Plan, Effective-k Graph, and Embeddings for the 5D Noise Variant

In [ ]:
plane4d_plan = compute_ot_plan(
    plane4d_data['obs0'],
    plane4d_data['obs1'],
    eps_times=higher_dim_eps_times,
    tol=tol,
    max_iter=max_iter,
)

print(f'orthogonal 4D-plane OT plan shape={plane4d_plan.shape}, total mass={plane4d_plan.sum():.6f}')
plot_matrix(
    plane4d_plan,
    title=f'Orthogonal 4D-plane/5D two-ring OT plan (eps_times={higher_dim_eps_times:g})',
    xlabel='batch 1 index',
    ylabel='batch 0 index',
    use_lognorm=True,
    figsize=(7.5, 5.5),
)

plane4d_four_block = build_ot_tsne_four_block_matrix(plane4d_plan)
plane4d_four_block_nnz = int(np.count_nonzero(plane4d_four_block))
print(
    f'orthogonal 4D-plane/5D 4-block matrix shape={plane4d_four_block.shape}, '
    f'nnz={plane4d_four_block_nnz}, '
    f'density={plane4d_four_block_nnz / plane4d_four_block.size:.4f}, '
    f'sum={plane4d_four_block.sum():.6f}'
)
plot_matrix(
    plane4d_four_block,
    title='Orthogonal 4D-plane/5D variant full 4-block matrix from the OT plan',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=True,
    figsize=(7.5, 5.5),
)

plane4d_sparse_four_block, plane4d_sparse_split = sparsify_four_block_top_k(
    plane4d_four_block,
    n0=n0,
    n1=n1,
    effective_k=higher_dim_effective_k,
    binarize=True,
)
plane4d_sparse_nnz = int(np.count_nonzero(plane4d_sparse_four_block))
print(
    f'orthogonal 4D-plane/5D effective-k sparse 4-block support: '
    f'effective_k={plane4d_sparse_split.effective_k} '
    f'({plane4d_sparse_split.batch0_k}+{plane4d_sparse_split.batch1_k}), '
    f'nnz={plane4d_sparse_nnz}, '
    f'density={plane4d_sparse_nnz / plane4d_sparse_four_block.size:.4f}, '
    f'sum={plane4d_sparse_four_block.sum():.6f}'
)
plot_matrix(
    plane4d_sparse_four_block,
    title=f'Orthogonal 4D-plane/5D effective-k support (k={higher_dim_effective_k})',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=False,
    figsize=(7.5, 5.5),
)

plane4d_step5_graph = normalize_ot_tsne_graph(plane4d_sparse_four_block)
plane4d_step5_nnz = int(np.count_nonzero(plane4d_step5_graph))
print(
    f'orthogonal 4D-plane/5D normalized effective-k graph: nnz={plane4d_step5_nnz}, '
    f'density={plane4d_step5_nnz / plane4d_step5_graph.size:.4f}, '
    f'sum={plane4d_step5_graph.sum():.6f}'
)
plot_matrix(
    plane4d_step5_graph,
    title='Orthogonal 4D-plane/5D normalized effective-k graph',
    xlabel='all points index',
    ylabel='all points index',
    use_lognorm=False,
    figsize=(7.5, 5.5),
)


In [ ]:
plane4d_ot_tsne_embedding = run_tsne_from_graph(
    plane4d_step5_graph,
    seed=tsne_seed,
    n_epochs=n_epochs,
)
plane4d_ot_tsne_metrics = metric_triplet(
    plane4d_data['clean0'],
    plane4d_data['clean1'],
    plane4d_ot_tsne_embedding,
    k_conc=k_conc,
    k_cross=k_cross,
)

print_metric_summary('Orthogonal 4D-plane OT-tSNE', plane4d_ot_tsne_metrics)
plot_embedding(
    plane4d_ot_tsne_embedding,
    plane4d_data['batch_ids'],
    plane4d_data['component_ids'],
    title=f'Orthogonal 4D-plane/5D OT-tSNE embedding\n(eps_times={higher_dim_eps_times:g}, k={higher_dim_effective_k})',
    metrics=plane4d_ot_tsne_metrics,
    seed=tsne_seed,
    component_names=PLANE4D_COMPONENT_NAMES,
)

plane4d_eot_embedding = compute_eot_embedding_from_plan(
    plane4d_plan,
    n0=n0,
    n1=n1,
    embd_dim=embd_dim,
    t=eot_t,
)
plane4d_eot_metrics = metric_triplet(
    plane4d_data['clean0'],
    plane4d_data['clean1'],
    plane4d_eot_embedding,
    k_conc=k_conc,
    k_cross=k_cross,
)

print_metric_summary(f'Orthogonal 4D-plane EOT t={eot_t}', plane4d_eot_metrics)
plane4d_clean_dim2 = np.concatenate([
    plane4d_data['clean0'][:, 1],
    plane4d_data['clean1'][:, 1],
])
plot_embedding(
    plane4d_eot_embedding,
    plane4d_data['batch_ids'],
    plane4d_data['component_ids'],
    title=f'Orthogonal 4D-plane/5D EOT embedding (t={eot_t})',
    metrics=plane4d_eot_metrics,
    seed=tsne_seed,
    display_mode='axiswise',
    display_note='axis-wise robust scaling for display only',
    component_names=PLANE4D_COMPONENT_NAMES,
    gradient_values=plane4d_clean_dim2,
    gradient_label='clean dim 2',
)

plane4d_batch_colors = np.array([BATCH_COLORS[int(v)] for v in plane4d_data['batch_ids']])
plane4d_perm = np.random.RandomState(tsne_seed).permutation(plane4d_batch_colors.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, emb, title, metrics, display_mode in (
    (axes[0], plane4d_ot_tsne_embedding, 'Orthogonal 4D-plane OT-tSNE', plane4d_ot_tsne_metrics, 'isotropic'),
    (axes[1], plane4d_eot_embedding, f'Orthogonal 4D-plane EOT t={eot_t}', plane4d_eot_metrics, 'axiswise'),
):
    emb_display = prepare_embedding_for_display(emb, mode=display_mode)
    ax.scatter(
        emb_display[plane4d_perm, 0],
        emb_display[plane4d_perm, 1],
        c=plane4d_batch_colors[plane4d_perm],
        s=3,
        alpha=0.35,
    )
    ax.set_title(title)
    ax.set_aspect('equal')
    apply_robust_display_limits(ax, emb_display, square=(display_mode == 'isotropic'))
    ax.axis('off')
    ax.text(
        0.02,
        0.98,
        metric_box_text(metrics),
        transform=ax.transAxes,
        va='top',
        ha='left',
        fontsize=8,
        bbox={'boxstyle': 'round,pad=0.2', 'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
    )

fig.suptitle(
    'Batch-colored comparison: orthogonal 4D-plane/5D OT-tSNE vs EOT t=0\n'
    '(EOT uses axis-wise display scaling only)',
    fontsize=14,
)
plt.show()

plot_eot_ring_diagnostics(
    plane4d_eot_embedding,
    n0=n0,
    component0=plane4d_data['component0'],
    component1=plane4d_data['component1'],
    seed=tsne_seed,
    t=eot_t,
    component_names=PLANE4D_COMPONENT_NAMES,
)


## EOT `eps_times` Sweep for the 2D and 3D Examples

To make the comparison transparent, this section gives EOT its own `eps_times` sweep. For each case we recompute the OT plan at `eps_times = [1, 10, 100, 300]`, build the EOT `t=0` embedding, and mark the candidate with the best concordance score. The grid shows the full sweep colored by ring; the selected panel below each grid is the best-concordance EOT solution from that sweep.

In [ ]:

import warnings


eot_sweep_eps_times = [1.0, 10.0, 100.0, 300.0]
eot_sweep_selection_metric = 'concordance'


def format_eps_value(value):
    return f'{float(value):g}'


def select_best_eot_sweep_result(results, metric_name=eot_sweep_selection_metric):
    if metric_name == 'foscttm':
        return min(results, key=lambda result: result['metrics'][metric_name])
    return max(results, key=lambda result: result['metrics'][metric_name])


def run_eot_eps_sweep(case_label, data_bundle, component_names):
    results = []
    for eps_value in eot_sweep_eps_times:
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter('always', UserWarning)
            sweep_plan = compute_ot_plan(
                data_bundle['obs0'],
                data_bundle['obs1'],
                eps_times=eps_value,
                tol=tol,
                max_iter=max_iter,
            )

        sweep_embedding = compute_eot_embedding_from_plan(
            sweep_plan,
            n0=n0,
            n1=n1,
            embd_dim=embd_dim,
            t=eot_t,
        )
        sweep_metrics = metric_triplet(
            data_bundle['clean0'],
            data_bundle['clean1'],
            sweep_embedding,
            k_conc=k_conc,
            k_cross=k_cross,
        )
        results.append(
            {
                'case_label': case_label,
                'eps_times': float(eps_value),
                'embedding': sweep_embedding,
                'metrics': sweep_metrics,
                'n_warnings': len(caught_warnings),
            }
        )

    best_concordance = select_best_eot_sweep_result(results, metric_name='concordance')
    best_cross_concordance = select_best_eot_sweep_result(results, metric_name='cross_concordance')
    best_foscttm = select_best_eot_sweep_result(results, metric_name='foscttm')

    print(f'{case_label} EOT eps_times sweep (t={eot_t})')
    for result in results:
        metrics = result['metrics']
        warning_note = '' if result['n_warnings'] == 0 else f", sinkhorn_warnings={result['n_warnings']}"
        print(
            f"  eps_times={format_eps_value(result['eps_times'])}: "
            f"conc={metrics['concordance']:.4f}, "
            f"x-conc={metrics['cross_concordance']:.4f}, "
            f"foscttm={metrics['foscttm']:.4f}"
            f'{warning_note}'
        )
    print(
        f"  best concordance: eps_times={format_eps_value(best_concordance['eps_times'])}; "
        f"best x-conc: eps_times={format_eps_value(best_cross_concordance['eps_times'])}; "
        f"best foscttm: eps_times={format_eps_value(best_foscttm['eps_times'])}"
    )
    print()
    return {
        'results': results,
        'best_concordance': best_concordance,
        'best_cross_concordance': best_cross_concordance,
        'best_foscttm': best_foscttm,
        'component_names': component_names,
        'data': data_bundle,
    }


def plot_eot_sweep_ring_grid(sweep, seed=0):
    results = sweep['results']
    data_bundle = sweep['data']
    component_names = sweep['component_names']
    best_eps = sweep['best_concordance']['eps_times']

    component_colors = np.array([COMPONENT_COLORS[int(v)] for v in data_bundle['component_ids']])
    perm = np.random.RandomState(seed).permutation(component_colors.shape[0])

    fig, axes = plt.subplots(1, len(results), figsize=(4.2 * len(results), 4.2), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, result in zip(axes, results):
        embedding_display = prepare_embedding_for_display(result['embedding'], mode='axiswise')
        ax.scatter(
            embedding_display[perm, 0],
            embedding_display[perm, 1],
            c=component_colors[perm],
            s=3,
            alpha=0.45,
        )
        best_note = ' (best conc)' if result['eps_times'] == best_eps else ''
        ax.set_title(f"eps_times={format_eps_value(result['eps_times'])}{best_note}")
        ax.set_aspect('equal')
        apply_robust_display_limits(ax, embedding_display, square=False)
        ax.axis('off')
        ax.text(
            0.02,
            0.98,
            metric_box_text(result['metrics']),
            transform=ax.transAxes,
            va='top',
            ha='left',
            fontsize=8,
            bbox={'boxstyle': 'round,pad=0.2', 'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none'},
        )

    axes[-1].legend(
        handles=[
            Line2D(
                [0],
                [0],
                marker='o',
                color='w',
                label=component_names.get(idx, f'component {idx}'),
                markerfacecolor=color,
                markersize=8,
            )
            for idx, color in COMPONENT_COLORS.items()
        ],
        loc='best',
    )
    fig.suptitle(
        f"{results[0]['case_label']} EOT t={eot_t} eps_times sweep\n"
        '(colored by ring, axis-wise display scaling)',
        fontsize=14,
    )
    plt.show()


def plot_best_eot_sweep_embedding(sweep, seed=0):
    best = sweep['best_concordance']
    data_bundle = sweep['data']
    plot_embedding(
        best['embedding'],
        data_bundle['batch_ids'],
        data_bundle['component_ids'],
        title=(
            f"{best['case_label']} best EOT t={eot_t} from eps_times sweep\n"
            f"(best concordance at eps_times={format_eps_value(best['eps_times'])})"
        ),
        metrics=best['metrics'],
        seed=seed,
        display_mode='axiswise',
        display_note='axis-wise robust scaling for display only',
        component_names=sweep['component_names'],
    )


eot_sweep_2d = run_eot_eps_sweep('2D two-ring', data, COMPONENT_NAMES)
plot_eot_sweep_ring_grid(eot_sweep_2d, seed=tsne_seed)
plot_best_eot_sweep_embedding(eot_sweep_2d, seed=tsne_seed)


eot_sweep_3d = run_eot_eps_sweep('3D floating-ring', floating_data, FLOATING_COMPONENT_NAMES)
plot_eot_sweep_ring_grid(eot_sweep_3d, seed=tsne_seed)
plot_best_eot_sweep_embedding(eot_sweep_3d, seed=tsne_seed)
